# Merge LoRA Adapter into a Full FLAN-T5 Model
This notebook loads the saved LoRA adapter from Google Drive, verifies the adapter folder contents, merges it into the base `google/flan-t5-base` model, and saves the full standalone model.

In [ ]:
!pip install -q --upgrade "torchao>=0.16.0" transformers datasets peft accelerate sentencepiece

In [ ]:
import importlib
import pkg_resources

for pkg in ["torchao", "peft", "transformers"]:
    try:
        module = importlib.import_module(pkg)
        print(pkg, module.__version__)
    except Exception as exc:
        print(pkg, "ERROR", exc)

# Verify package requirements
assert pkg_resources.parse_version(importlib.import_module("torchao").__version__) >= pkg_resources.parse_version("0.16.0"), \
    "torchao must be >= 0.16.0"

print('Dependency versions are correct. Restart the runtime if necessary before running the next cells.')

In [ ]:
import os
import torch
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

drive.mount('/content/drive')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'CUDA available: {torch.cuda.is_available()} | Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
model_name = 'google/flan-t5-base'
adapter_dir = '/content/drive/MyDrive/flan_t5_lora_adapter_base'
merged_dir = '/content/drive/MyDrive/flan_t5_lora_merged_base'

print('Adapter directory exists:', os.path.exists(adapter_dir))
if os.path.exists(adapter_dir):
    print('Adapter directory contents:')
    for path in sorted(os.listdir(adapter_dir)):
        print('-', path)
else:
    raise FileNotFoundError(f'Adapter directory not found: {adapter_dir}')

In [ ]:
# Load tokenizer and base model
# If Hugging Face warns about tied embeddings, we disable it on the config.
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
base_model.config.tie_word_embeddings = False
base_model.to(device)
print(f'Base model loaded: {model_name}')

In [ ]:
# Load the LoRA adapter and merge it into the base model
peft_model = PeftModel.from_pretrained(base_model, adapter_dir)
merged_model = peft_model.merge_and_unload()
merged_model.to(device)
print('Merged model ready')

In [ ]:
# Save the final merged model and tokenizer
merged_model.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)
print(f'Full merged model saved to: {merged_dir}')

In [ ]:
# Quick inference check
# Replace this text with any sentence or paragraph you want to simplify.
sample_text = input('Enter text to simplify: ') if False else (
    'Wellington had deployed them on the reverse slope of the ridge, '
    'where they could neither be easily seen nor easily softened up with artillery.'
)

prompt = f'Simplify the following text:\n{sample_text}'
inputs = tokenizer(prompt, return_tensors='pt', max_length=512, truncation=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = merged_model.generate(
        **inputs,
        max_new_tokens=256,
        num_beams=4,
        repetition_penalty=1.5,
        no_repeat_ngram_size=3,
        early_stopping=True,
    )

print('Original:  ', sample_text)
print('Simplified:', tokenizer.decode(outputs[0], skip_special_tokens=True))